# End-of-Text Attender

Attends primarily to the beginning-of-sequence / end-of-text token

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables, show_attention_to_position, TEXT,
)

/home/burny/projects/ai/mechinterp/attention-head-zoo-2-layer-attention-only-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)

## EOT Attention % Across All 24 Heads

Mean fraction of attention weight allocated to position 0 (end-of-text token), averaged across all destination positions. Sorted by raw % descending.

In [3]:
show_attention_to_position(cache, position=0, label="EOT")

Head            EOT %  Level
-----------------------------------
L1H4           89.74%  fullish
L1H10          82.98%  fullish
L0H3           82.33%  fullish
L1H8           37.91%  partial
L1H3           37.12%  partial
L0H6           32.60%  partial
L1H1           29.33%  partial
L1H2           28.60%  partial
L0H2           26.97%  partial
L1H11          26.46%  partial
L0H1           24.58%  partial
L1H0           24.31%  partial
L0H5           21.36%  partial
L0H11          20.44%  partial
L0H0           20.11%  partial
L1H6           17.48%  partial
L1H9           16.82%  partial
L0H8           16.76%  partial
L0H10          15.04%  partial
L0H4           12.33%  partial
L0H9           11.06%  partial
L1H5            8.17%  almost none
L1H7            3.65%  almost none
L0H7            3.23%  almost none


In [4]:
pct = get_attention_pattern(cache, layer=1, head=4)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L1H4 — {pct:.2f}% ({level})\n\nCares only about end of text token"))

---
## L1H4 — 89.74% (fullish)

Cares only about end of text token

In [5]:
show_head_pattern(str_tokens, cache, layer=1, head=4)

In [6]:
attention = get_attention_pattern(cache, layer=1, head=4)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.9898 |
| 3 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.9856 |
| 4 | ` level` | `<\|endoftext\|>` | 32 | 0 | 0.9854 |
| 5 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.9847 |
| 6 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.9813 |
| 7 | ` up` | `<\|endoftext\|>` | 29 | 0 | 0.9809 |
| 8 | ` century` | `<\|endoftext\|>` | 20 | 0 | 0.9780 |
| 9 | ` is` | `<\|endoftext\|>` | 11 | 0 | 0.9774 |
| 10 | ` learning` | `<\|endoftext\|>` | 25 | 0 | 0.9773 |
| 11 | ` solid` | `<\|endoftext\|>` | 52 | 0 | 0.9761 |
| 12 | `.` | `<\|endoftext\|>` | 21 | 0 | 0.9760 |
| 13 | ` techniques` | `<\|endoftext\|>` | 26 | 0 | 0.9744 |
| 14 | ` scaled` | `<\|endoftext\|>` | 28 | 0 | 0.9737 |
| 15 | ` systems` | `<\|endoftext\|>` | 41 | 0 | 0.9707 |
| 16 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.9623 |
| 17 | ` to` | `<\|endoftext\|>` | 30 | 0 | 0.9620 |
| 18 | ` manip` | `<\|endoftext\|>` | 46 | 0 | 0.9586 |
| 19 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.9568 |
| 20 | ` default` | `<\|endoftext\|>` | 39 | 0 | 0.9496 |
| 21 | ` current` | `<\|endoftext\|>` | 23 | 0 | 0.9480 |
| 22 | ` they` | `<\|endoftext\|>` | 36 | 0 | 0.9478 |
| 23 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.9436 |
| 24 | ` this` | `<\|endoftext\|>` | 19 | 0 | 0.9416 |
| 25 | `,` | `<\|endoftext\|>` | 33 | 0 | 0.9398 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` by` | ` techniques` | 38 | 26 | 0.0000 |
| 2 | ` to` | ` learning` | 30 | 25 | 0.0000 |
| 3 | ` no` | ` intelligence` | 51 | 10 | 0.0000 |
| 4 | ` solid` | ` than` | 52 | 14 | 0.0000 |
| 5 | ` that` | ` learning` | 50 | 25 | 0.0000 |
| 6 | ` that` | ` intelligence` | 50 | 10 | 0.0000 |
| 7 | ` manip` | ` machine` | 46 | 24 | 0.0000 |
| 8 | ` that` | `human` | 50 | 8 | 0.0000 |
| 9 | ` no` | ` learning` | 51 | 25 | 0.0000 |
| 10 | `.` | `ulative` | 61 | 47 | 0.0000 |
| 11 | ` to` | ` techniques` | 30 | 26 | 0.0000 |
| 12 | `,` | ` techniques` | 48 | 26 | 0.0000 |
| 13 | ` solid` | ` would` | 52 | 37 | 0.0000 |
| 14 | ` or` | ` machine` | 45 | 24 | 0.0000 |
| 15 | ` to` | ` intelligence` | 30 | 10 | 0.0000 |
| 16 | ` would` | ` learning` | 37 | 25 | 0.0000 |
| 17 | ` by` | ` learning` | 38 | 25 | 0.0000 |
| 18 | ` is` | ` intelligence` | 11 | 10 | 0.0000 |
| 19 | ` avoid` | ` learning` | 59 | 25 | 0.0000 |
| 20 | ` manip` | ` level` | 46 | 32 | 0.0000 |
| 21 | ` than` | ` intelligence` | 14 | 10 | 0.0000 |
| 22 | ` to` | `human` | 30 | 8 | 0.0000 |
| 23 | ` scaled` | ` intelligence` | 28 | 10 | 0.0000 |
| 24 | ` scaled` | ` learning` | 28 | 25 | 0.0000 |
| 25 | ` that` | ` learning` | 42 | 25 | 0.0000 |

In [7]:
pct = get_attention_pattern(cache, layer=0, head=3)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L0H3 — {pct:.2f}% ({level})\n\nCares only about end of text token"))

---
## L0H3 — 82.33% (fullish)

Cares only about end of text token

In [8]:
show_head_pattern(str_tokens, cache, layer=0, head=3)

In [9]:
attention = get_attention_pattern(cache, layer=0, head=3)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.9974 |
| 3 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.9894 |
| 4 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.9892 |
| 5 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.9860 |
| 6 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.9708 |
| 7 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.9694 |
| 8 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.9678 |
| 9 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.9658 |
| 10 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.9646 |
| 11 | ` is` | `<\|endoftext\|>` | 11 | 0 | 0.9445 |
| 12 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.9434 |
| 13 | ` century` | `<\|endoftext\|>` | 20 | 0 | 0.9389 |
| 14 | ` likely` | `<\|endoftext\|>` | 13 | 0 | 0.9300 |
| 15 | ` created` | `<\|endoftext\|>` | 18 | 0 | 0.9274 |
| 16 | ` not` | `<\|endoftext\|>` | 15 | 0 | 0.9134 |
| 17 | `.` | `<\|endoftext\|>` | 21 | 0 | 0.9115 |
| 18 | ` than` | `<\|endoftext\|>` | 14 | 0 | 0.9106 |
| 19 | ` more` | `<\|endoftext\|>` | 12 | 0 | 0.9089 |
| 20 | ` current` | `<\|endoftext\|>` | 23 | 0 | 0.9065 |
| 21 | ` this` | `<\|endoftext\|>` | 19 | 0 | 0.9043 |
| 22 | ` to` | `<\|endoftext\|>` | 16 | 0 | 0.9024 |
| 23 | ` be` | `<\|endoftext\|>` | 17 | 0 | 0.9022 |
| 24 | ` machine` | `<\|endoftext\|>` | 24 | 0 | 0.8755 |
| 25 | ` learning` | `<\|endoftext\|>` | 25 | 0 | 0.8745 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` known` | ` known` | 55 | 55 | 0.0006 |
| 2 | ` known` | ` not` | 55 | 15 | 0.0008 |
| 3 | ` plans` | ` not` | 53 | 15 | 0.0010 |
| 4 | `.` | `We` | 61 | 1 | 0.0010 |
| 5 | ` century` | `We` | 20 | 1 | 0.0010 |
| 6 | ` known` | ` is` | 55 | 11 | 0.0010 |
| 7 | ` current` | ` not` | 23 | 15 | 0.0010 |
| 8 | `ulative` | ` not` | 47 | 15 | 0.0010 |
| 9 | ` how` | ` not` | 57 | 15 | 0.0011 |
| 10 | ` level` | ` not` | 32 | 15 | 0.0012 |
| 11 | ` this` | ` known` | 60 | 55 | 0.0012 |
| 12 | ` plans` | ` is` | 53 | 11 | 0.0012 |
| 13 | `.` | `We` | 21 | 1 | 0.0012 |
| 14 | ` for` | ` not` | 56 | 15 | 0.0012 |
| 15 | ` powerful` | `We` | 4 | 1 | 0.0012 |
| 16 | ` or` | ` not` | 45 | 15 | 0.0012 |
| 17 | `human` | `We` | 8 | 1 | 0.0013 |
| 18 | ` known` | `We` | 55 | 1 | 0.0013 |
| 19 | ` systems` | ` not` | 41 | 15 | 0.0013 |
| 20 | ` techniques` | ` not` | 26 | 15 | 0.0013 |
| 21 | ` known` | ` up` | 55 | 29 | 0.0013 |
| 22 | ` century` | ` is` | 20 | 11 | 0.0013 |
| 23 | ` learning` | ` significantly` | 25 | 6 | 0.0013 |
| 24 | ` deceptive` | ` not` | 44 | 15 | 0.0013 |
| 25 | ` learning` | ` not` | 25 | 15 | 0.0013 |

In [10]:
pct = get_attention_pattern(cache, layer=0, head=6)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L0H6 — {pct:.2f}% ({level})\n\nAttends to only end of text token"))

---
## L0H6 — 32.60% (partial)

Attends to only end of text token

In [11]:
show_head_pattern(str_tokens, cache, layer=0, head=6)

In [12]:
attention = get_attention_pattern(cache, layer=0, head=6)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.8699 |
| 3 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.8603 |
| 4 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.8541 |
| 5 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.8325 |
| 6 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.8020 |
| 7 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.7745 |
| 8 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.7326 |
| 9 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.6470 |
| 10 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.6307 |
| 11 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.5851 |
| 12 | ` likely` | `<\|endoftext\|>` | 13 | 0 | 0.5529 |
| 13 | ` more` | `<\|endoftext\|>` | 12 | 0 | 0.5387 |
| 14 | ` century` | `<\|endoftext\|>` | 20 | 0 | 0.4885 |
| 15 | ` not` | `<\|endoftext\|>` | 15 | 0 | 0.4581 |
| 16 | ` is` | `<\|endoftext\|>` | 11 | 0 | 0.4465 |
| 17 | ` created` | `<\|endoftext\|>` | 18 | 0 | 0.4464 |
| 18 | ` this` | `<\|endoftext\|>` | 19 | 0 | 0.4031 |
| 19 | ` than` | `<\|endoftext\|>` | 14 | 0 | 0.3900 |
| 20 | ` to` | `<\|endoftext\|>` | 16 | 0 | 0.3796 |
| 21 | `We` | `We` | 1 | 1 | 0.3693 |
| 22 | ` be` | `<\|endoftext\|>` | 17 | 0 | 0.3594 |
| 23 | ` learning` | `<\|endoftext\|>` | 25 | 0 | 0.3473 |
| 24 | `.` | `<\|endoftext\|>` | 21 | 0 | 0.3053 |
| 25 | ` default` | `<\|endoftext\|>` | 39 | 0 | 0.3004 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `ulative` | ` we` | 47 | 34 | 0.0008 |
| 2 | `ulative` | `We` | 47 | 1 | 0.0013 |
| 3 | `.` | ` more` | 61 | 12 | 0.0013 |
| 4 | ` are` | `.` | 54 | 21 | 0.0015 |
| 5 | ` manip` | `.` | 46 | 21 | 0.0015 |
| 6 | ` known` | ` than` | 55 | 14 | 0.0016 |
| 7 | ` deceptive` | `.` | 44 | 21 | 0.0016 |
| 8 | `ulative` | ` than` | 47 | 14 | 0.0017 |
| 9 | ` known` | ` and` | 55 | 49 | 0.0018 |
| 10 | ` for` | ` than` | 56 | 14 | 0.0018 |
| 11 | `.` | ` they` | 61 | 36 | 0.0018 |
| 12 | `.` | ` super` | 61 | 7 | 0.0018 |
| 13 | ` are` | `.` | 43 | 21 | 0.0019 |
| 14 | ` for` | `.` | 56 | 21 | 0.0019 |
| 15 | `ulative` | `.` | 47 | 21 | 0.0020 |
| 16 | `ulative` | ` they` | 47 | 36 | 0.0020 |
| 17 | ` systems` | ` than` | 41 | 14 | 0.0020 |
| 18 | ` known` | ` super` | 55 | 7 | 0.0020 |
| 19 | ` are` | ` super` | 54 | 7 | 0.0020 |
| 20 | ` systems` | `.` | 41 | 21 | 0.0020 |
| 21 | ` current` | `.` | 23 | 21 | 0.0021 |
| 22 | ` avoid` | `.` | 59 | 21 | 0.0021 |
| 23 | ` how` | `.` | 57 | 21 | 0.0021 |
| 24 | `,` | `We` | 48 | 1 | 0.0022 |
| 25 | `ulative` | ` be` | 47 | 17 | 0.0022 |

In [13]:
pct = get_attention_pattern(cache, layer=1, head=3)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L1H3 — {pct:.2f}% ({level})\n\nPrimarily end of text token, diffuse across positions"))

---
## L1H3 — 37.12% (partial)

Primarily end of text token, diffuse across positions

In [14]:
show_head_pattern(str_tokens, cache, layer=1, head=3)

In [15]:
attention = get_attention_pattern(cache, layer=1, head=3)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` more` | `<\|endoftext\|>` | 12 | 0 | 0.8894 |
| 3 | ` were` | `<\|endoftext\|>` | 27 | 0 | 0.8078 |
| 4 | ` is` | `<\|endoftext\|>` | 11 | 0 | 0.7239 |
| 5 | ` are` | `<\|endoftext\|>` | 43 | 0 | 0.7235 |
| 6 | ` to` | `<\|endoftext\|>` | 30 | 0 | 0.7224 |
| 7 | ` this` | `<\|endoftext\|>` | 31 | 0 | 0.6553 |
| 8 | ` likely` | `<\|endoftext\|>` | 13 | 0 | 0.6150 |
| 9 | ` this` | `<\|endoftext\|>` | 19 | 0 | 0.6107 |
| 10 | ` than` | `<\|endoftext\|>` | 14 | 0 | 0.5941 |
| 11 | `We` | `We` | 1 | 1 | 0.5915 |
| 12 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.5839 |
| 13 | ` be` | `<\|endoftext\|>` | 17 | 0 | 0.5811 |
| 14 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.5658 |
| 15 | ` up` | `<\|endoftext\|>` | 29 | 0 | 0.5482 |
| 16 | ` are` | `<\|endoftext\|>` | 54 | 0 | 0.5447 |
| 17 | ` think` | `<\|endoftext\|>` | 35 | 0 | 0.5401 |
| 18 | ` created` | `<\|endoftext\|>` | 18 | 0 | 0.5282 |
| 19 | ` century` | `<\|endoftext\|>` | 20 | 0 | 0.5235 |
| 20 | ` not` | `<\|endoftext\|>` | 15 | 0 | 0.4779 |
| 21 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.4618 |
| 22 | ` level` | `<\|endoftext\|>` | 32 | 0 | 0.4509 |
| 23 | ` this` | `<\|endoftext\|>` | 60 | 0 | 0.4431 |
| 24 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.4328 |
| 25 | ` If` | `<\|endoftext\|>` | 22 | 0 | 0.4152 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `.` | ` up` | 61 | 29 | 0.0000 |
| 2 | `.` | ` known` | 61 | 55 | 0.0000 |
| 3 | ` no` | ` up` | 51 | 29 | 0.0000 |
| 4 | `.` | ` created` | 61 | 18 | 0.0001 |
| 5 | ` are` | ` up` | 43 | 29 | 0.0001 |
| 6 | `.` | ` scaled` | 61 | 28 | 0.0001 |
| 7 | ` no` | ` created` | 51 | 18 | 0.0001 |
| 8 | ` no` | ` likely` | 51 | 13 | 0.0001 |
| 9 | `.` | ` likely` | 61 | 13 | 0.0001 |
| 10 | ` solid` | ` up` | 52 | 29 | 0.0001 |
| 11 | `,` | ` up` | 48 | 29 | 0.0001 |
| 12 | ` no` | ` scaled` | 51 | 28 | 0.0001 |
| 13 | ` think` | ` up` | 35 | 29 | 0.0001 |
| 14 | ` are` | ` up` | 54 | 29 | 0.0001 |
| 15 | ` or` | ` techniques` | 45 | 26 | 0.0001 |
| 16 | ` solid` | ` created` | 52 | 18 | 0.0001 |
| 17 | ` no` | ` think` | 51 | 35 | 0.0002 |
| 18 | ` than` | ` likely` | 14 | 13 | 0.0002 |
| 19 | `.` | ` think` | 61 | 35 | 0.0002 |
| 20 | ` that` | ` up` | 50 | 29 | 0.0002 |
| 21 | ` solid` | ` scaled` | 52 | 28 | 0.0003 |
| 22 | `,` | ` scaled` | 48 | 28 | 0.0003 |
| 23 | ` or` | ` up` | 45 | 29 | 0.0003 |
| 24 | ` for` | ` up` | 56 | 29 | 0.0003 |
| 25 | ` think` | ` created` | 35 | 18 | 0.0003 |

In [16]:
pct = get_attention_pattern(cache, layer=1, head=2)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L1H2 — {pct:.2f}% ({level})\n\nEnd of text, self-attention, content words attend to EOT"))

---
## L1H2 — 28.60% (partial)

End of text, self-attention, content words attend to EOT

In [17]:
show_head_pattern(str_tokens, cache, layer=1, head=2)

In [18]:
attention = get_attention_pattern(cache, layer=1, head=2)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` this` | ` this` | 19 | 19 | 0.9580 |
| 3 | ` manip` | `<\|endoftext\|>` | 46 | 0 | 0.8808 |
| 4 | ` deceptive` | `<\|endoftext\|>` | 44 | 0 | 0.8561 |
| 5 | ` current` | `<\|endoftext\|>` | 23 | 0 | 0.8513 |
| 6 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.7895 |
| 7 | ` this` | ` this` | 31 | 31 | 0.7619 |
| 8 | `,` | `,` | 5 | 5 | 0.7616 |
| 9 | ` how` | ` how` | 57 | 57 | 0.7222 |
| 10 | ` default` | `<\|endoftext\|>` | 39 | 0 | 0.6598 |
| 11 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.6597 |
| 12 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.6502 |
| 13 | ` think` | ` think` | 2 | 2 | 0.6331 |
| 14 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.6014 |
| 15 | `ulative` | `<\|endoftext\|>` | 47 | 0 | 0.5844 |
| 16 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.5542 |
| 17 | ` up` | `<\|endoftext\|>` | 29 | 0 | 0.5429 |
| 18 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.5312 |
| 19 | ` century` | ` this` | 20 | 19 | 0.5141 |
| 20 | ` level` | `<\|endoftext\|>` | 32 | 0 | 0.4990 |
| 21 | ` solid` | ` no` | 52 | 51 | 0.4897 |
| 22 | ` likely` | ` more` | 13 | 12 | 0.4820 |
| 23 | ` that` | ` that` | 3 | 3 | 0.4751 |
| 24 | ` no` | ` no` | 51 | 51 | 0.4425 |
| 25 | ` created` | `<\|endoftext\|>` | 18 | 0 | 0.4220 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` how` | ` up` | 57 | 29 | 0.0000 |
| 2 | ` how` | ` scaled` | 57 | 28 | 0.0000 |
| 3 | ` solid` | ` scaled` | 52 | 28 | 0.0000 |
| 4 | ` how` | ` think` | 57 | 35 | 0.0000 |
| 5 | ` solid` | ` up` | 52 | 29 | 0.0000 |
| 6 | ` manip` | ` significantly` | 46 | 6 | 0.0000 |
| 7 | ` solid` | ` created` | 52 | 18 | 0.0000 |
| 8 | ` this` | ` significantly` | 19 | 6 | 0.0000 |
| 9 | ` solid` | ` likely` | 52 | 13 | 0.0000 |
| 10 | ` solid` | ` century` | 52 | 20 | 0.0000 |
| 11 | ` how` | ` created` | 57 | 18 | 0.0000 |
| 12 | ` this` | ` significantly` | 60 | 6 | 0.0000 |
| 13 | ` this` | ` significantly` | 31 | 6 | 0.0000 |
| 14 | ` this` | ` scaled` | 60 | 28 | 0.0000 |
| 15 | ` manip` | ` century` | 46 | 20 | 0.0000 |
| 16 | ` solid` | ` significantly` | 52 | 6 | 0.0000 |
| 17 | ` are` | `,` | 54 | 33 | 0.0000 |
| 18 | ` default` | ` scaled` | 39 | 28 | 0.0001 |
| 19 | ` this` | ` techniques` | 31 | 26 | 0.0001 |
| 20 | ` solid` | ` think` | 52 | 35 | 0.0001 |
| 21 | ` level` | ` scaled` | 32 | 28 | 0.0001 |
| 22 | ` are` | `,` | 54 | 48 | 0.0001 |
| 23 | ` solid` | ` be` | 52 | 17 | 0.0001 |
| 24 | ` how` | ` solid` | 57 | 52 | 0.0001 |
| 25 | ` how` | ` likely` | 57 | 13 | 0.0001 |

In [19]:
pct = get_attention_pattern(cache, layer=1, head=11)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L1H11 — {pct:.2f}% ({level})\n\nEnd of text for function words, glue words, previous token"))

---
## L1H11 — 26.46% (partial)

End of text for function words, glue words, previous token

In [20]:
show_head_pattern(str_tokens, cache, layer=1, head=11)

In [21]:
attention = get_attention_pattern(cache, layer=1, head=11)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` would` | `<\|endoftext\|>` | 37 | 0 | 0.9985 |
| 3 | ` we` | `<\|endoftext\|>` | 34 | 0 | 0.9985 |
| 4 | ` they` | `<\|endoftext\|>` | 36 | 0 | 0.9845 |
| 5 | ` to` | `<\|endoftext\|>` | 16 | 0 | 0.9825 |
| 6 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.9767 |
| 7 | ` to` | `<\|endoftext\|>` | 58 | 0 | 0.9674 |
| 8 | ` that` | `<\|endoftext\|>` | 42 | 0 | 0.8708 |
| 9 | ` think` | `We` | 2 | 1 | 0.7966 |
| 10 | ` that` | `<\|endoftext\|>` | 50 | 0 | 0.7939 |
| 11 | `,` | ` If` | 33 | 22 | 0.7250 |
| 12 | ` and` | `<\|endoftext\|>` | 49 | 0 | 0.6742 |
| 13 | ` or` | `<\|endoftext\|>` | 45 | 0 | 0.6131 |
| 14 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.5327 |
| 15 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.5080 |
| 16 | ` to` | `<\|endoftext\|>` | 30 | 0 | 0.4881 |
| 17 | ` not` | `<\|endoftext\|>` | 15 | 0 | 0.4855 |
| 18 | ` powerful` | `We` | 4 | 1 | 0.4554 |
| 19 | ` likely` | ` more` | 13 | 12 | 0.4150 |
| 20 | `,` | ` If` | 48 | 22 | 0.4147 |
| 21 | ` techniques` | `<\|endoftext\|>` | 26 | 0 | 0.3999 |
| 22 | ` default` | ` by` | 39 | 38 | 0.3800 |
| 23 | `,` | ` that` | 5 | 3 | 0.3781 |
| 24 | `human` | `human` | 8 | 8 | 0.3453 |
| 25 | `,` | `,` | 5 | 5 | 0.3346 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` would` | ` century` | 37 | 20 | 0.0000 |
| 2 | ` to` | ` scaled` | 58 | 28 | 0.0000 |
| 3 | ` would` | ` created` | 37 | 18 | 0.0000 |
| 4 | ` would` | ` level` | 37 | 32 | 0.0000 |
| 5 | ` to` | ` up` | 58 | 29 | 0.0000 |
| 6 | ` would` | ` scaled` | 37 | 28 | 0.0000 |
| 7 | ` would` | ` techniques` | 37 | 26 | 0.0000 |
| 8 | ` would` | ` up` | 37 | 29 | 0.0000 |
| 9 | ` that` | ` scaled` | 50 | 28 | 0.0000 |
| 10 | ` would` | ` likely` | 37 | 13 | 0.0000 |
| 11 | ` that` | ` created` | 50 | 18 | 0.0000 |
| 12 | ` would` | ` intelligence` | 37 | 10 | 0.0000 |
| 13 | ` to` | ` created` | 58 | 18 | 0.0000 |
| 14 | ` would` | ` significantly` | 37 | 6 | 0.0000 |
| 15 | ` that` | ` up` | 50 | 29 | 0.0000 |
| 16 | ` would` | ` machine` | 37 | 24 | 0.0000 |
| 17 | ` to` | ` think` | 58 | 35 | 0.0000 |
| 18 | ` would` | ` learning` | 37 | 25 | 0.0000 |
| 19 | ` would` | ` machine` | 37 | 9 | 0.0000 |
| 20 | ` would` | ` were` | 37 | 27 | 0.0000 |
| 21 | ` that` | ` likely` | 50 | 13 | 0.0000 |
| 22 | ` to` | ` known` | 58 | 55 | 0.0000 |
| 23 | ` to` | ` likely` | 58 | 13 | 0.0000 |
| 24 | ` would` | ` think` | 37 | 35 | 0.0000 |
| 25 | ` would` | ` current` | 37 | 23 | 0.0000 |

In [22]:
pct = get_attention_pattern(cache, layer=0, head=2)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L0H2 — {pct:.2f}% ({level})\n\nCares primarily about end of text token and some ,"))

---
## L0H2 — 26.97% (partial)

Cares primarily about end of text token and some ,

In [23]:
show_head_pattern(str_tokens, cache, layer=0, head=2)

In [24]:
attention = get_attention_pattern(cache, layer=0, head=2)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.9749 |
| 3 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.9376 |
| 4 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.8972 |
| 5 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.8224 |
| 6 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.8152 |
| 7 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.7773 |
| 8 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.7680 |
| 9 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.7377 |
| 10 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.7195 |
| 11 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.6613 |
| 12 | ` likely` | `<\|endoftext\|>` | 13 | 0 | 0.5708 |
| 13 | ` century` | `<\|endoftext\|>` | 20 | 0 | 0.4431 |
| 14 | ` learning` | `.` | 25 | 21 | 0.3997 |
| 15 | ` techniques` | `.` | 26 | 21 | 0.3802 |
| 16 | ` be` | `<\|endoftext\|>` | 17 | 0 | 0.3629 |
| 17 | ` machine` | ` If` | 24 | 22 | 0.3160 |
| 18 | ` techniques` | ` If` | 26 | 22 | 0.3154 |
| 19 | ` think` | `<\|endoftext\|>` | 35 | 0 | 0.3022 |
| 20 | ` is` | `<\|endoftext\|>` | 11 | 0 | 0.2968 |
| 21 | ` level` | ` If` | 32 | 22 | 0.2953 |
| 22 | ` created` | `<\|endoftext\|>` | 18 | 0 | 0.2925 |
| 23 | ` more` | `<\|endoftext\|>` | 12 | 0 | 0.2899 |
| 24 | ` deceptive` | `<\|endoftext\|>` | 44 | 0 | 0.2720 |
| 25 | ` for` | ` known` | 56 | 55 | 0.2718 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` for` | `,` | 56 | 5 | 0.0000 |
| 2 | ` for` | ` to` | 56 | 16 | 0.0001 |
| 3 | ` plans` | ` more` | 53 | 12 | 0.0001 |
| 4 | ` plans` | ` significantly` | 53 | 6 | 0.0001 |
| 5 | ` systems` | ` significantly` | 41 | 6 | 0.0001 |
| 6 | ` for` | ` than` | 56 | 14 | 0.0001 |
| 7 | ` plans` | ` machine` | 53 | 9 | 0.0001 |
| 8 | ` systems` | ` more` | 41 | 12 | 0.0002 |
| 9 | ` for` | ` to` | 56 | 30 | 0.0002 |
| 10 | ` for` | `,` | 56 | 33 | 0.0002 |
| 11 | ` techniques` | ` significantly` | 26 | 6 | 0.0002 |
| 12 | ` for` | `.` | 56 | 21 | 0.0002 |
| 13 | ` and` | `,` | 49 | 5 | 0.0002 |
| 14 | ` systems` | ` machine` | 41 | 9 | 0.0002 |
| 15 | `.` | `,` | 61 | 5 | 0.0002 |
| 16 | ` plans` | ` think` | 53 | 2 | 0.0003 |
| 17 | ` to` | `,` | 58 | 5 | 0.0003 |
| 18 | `.` | `.` | 61 | 21 | 0.0003 |
| 19 | `.` | ` than` | 61 | 14 | 0.0003 |
| 20 | `,` | `,` | 48 | 5 | 0.0003 |
| 21 | ` known` | ` more` | 55 | 12 | 0.0003 |
| 22 | ` to` | ` be` | 58 | 17 | 0.0003 |
| 23 | ` for` | ` is` | 56 | 11 | 0.0003 |
| 24 | ` plans` | `human` | 53 | 8 | 0.0003 |
| 25 | ` to` | ` to` | 58 | 16 | 0.0003 |

In [25]:
pct = get_attention_pattern(cache, layer=0, head=10)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L0H10 — {pct:.2f}% ({level})\n\nGlue words, certainty, end of text"))

---
## L0H10 — 15.04% (partial)

Glue words, certainty, end of text

In [26]:
show_head_pattern(str_tokens, cache, layer=0, head=10)

In [27]:
attention = get_attention_pattern(cache, layer=0, head=10)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.9309 |
| 3 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.7291 |
| 4 | ` were` | ` If` | 27 | 22 | 0.6617 |
| 5 | ` techniques` | ` If` | 26 | 22 | 0.5650 |
| 6 | ` machine` | ` If` | 24 | 22 | 0.5593 |
| 7 | ` learning` | ` If` | 25 | 22 | 0.4936 |
| 8 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.4760 |
| 9 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.4380 |
| 10 | ` is` | `<\|endoftext\|>` | 11 | 0 | 0.4300 |
| 11 | `,` | ` If` | 33 | 22 | 0.4163 |
| 12 | ` current` | ` If` | 23 | 22 | 0.4105 |
| 13 | ` are` | ` no` | 54 | 51 | 0.3977 |
| 14 | ` to` | ` scaled` | 30 | 28 | 0.3950 |
| 15 | ` If` | ` If` | 22 | 22 | 0.3778 |
| 16 | ` scaled` | ` were` | 28 | 27 | 0.3550 |
| 17 | ` plans` | ` no` | 53 | 51 | 0.3476 |
| 18 | ` no` | ` that` | 51 | 50 | 0.3366 |
| 19 | ` for` | ` plans` | 56 | 53 | 0.3282 |
| 20 | ` to` | ` how` | 58 | 57 | 0.3281 |
| 21 | ` more` | ` is` | 12 | 11 | 0.3258 |
| 22 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.3243 |
| 23 | ` deceptive` | ` are` | 44 | 43 | 0.3203 |
| 24 | ` or` | ` are` | 45 | 43 | 0.3138 |
| 25 | ` that` | ` that` | 42 | 42 | 0.3105 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` plans` | ` intelligence` | 53 | 10 | 0.0000 |
| 2 | ` for` | `human` | 56 | 8 | 0.0000 |
| 3 | ` plans` | `human` | 53 | 8 | 0.0000 |
| 4 | ` to` | ` super` | 58 | 7 | 0.0000 |
| 5 | ` plans` | ` super` | 53 | 7 | 0.0000 |
| 6 | ` plans` | ` machine` | 53 | 9 | 0.0000 |
| 7 | ` for` | ` super` | 56 | 7 | 0.0000 |
| 8 | ` to` | `human` | 58 | 8 | 0.0000 |
| 9 | ` to` | ` machine` | 58 | 9 | 0.0000 |
| 10 | ` to` | ` intelligence` | 58 | 10 | 0.0000 |
| 11 | ` known` | `human` | 55 | 8 | 0.0000 |
| 12 | ` known` | ` machine` | 55 | 9 | 0.0000 |
| 13 | ` known` | ` super` | 55 | 7 | 0.0000 |
| 14 | ` plans` | ` powerful` | 53 | 4 | 0.0000 |
| 15 | ` are` | ` machine` | 54 | 9 | 0.0000 |
| 16 | ` known` | ` intelligence` | 55 | 10 | 0.0000 |
| 17 | ` known` | `,` | 55 | 5 | 0.0000 |
| 18 | ` known` | ` powerful` | 55 | 4 | 0.0000 |
| 19 | `.` | ` super` | 61 | 7 | 0.0000 |
| 20 | ` plans` | `,` | 53 | 5 | 0.0000 |
| 21 | ` solid` | `human` | 52 | 8 | 0.0000 |
| 22 | ` this` | ` super` | 60 | 7 | 0.0000 |
| 23 | ` for` | ` intelligence` | 56 | 10 | 0.0000 |
| 24 | ` are` | ` super` | 54 | 7 | 0.0000 |
| 25 | ` to` | `,` | 58 | 5 | 0.0000 |

In [28]:
pct = get_attention_pattern(cache, layer=0, head=11)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L0H11 — {pct:.2f}% ({level})\n\nTo itself, glue words, end of text, semantically salient (scaled up, deceptive), certainty"))

---
## L0H11 — 20.44% (partial)

To itself, glue words, end of text, semantically salient (scaled up, deceptive), certainty

In [29]:
show_head_pattern(str_tokens, cache, layer=0, head=11)

In [30]:
attention = get_attention_pattern(cache, layer=0, head=11)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.9864 |
| 3 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.9362 |
| 4 | ` deceptive` | ` deceptive` | 44 | 44 | 0.9251 |
| 5 | ` manip` | ` manip` | 46 | 46 | 0.9240 |
| 6 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.8913 |
| 7 | ` up` | ` scaled` | 29 | 28 | 0.8883 |
| 8 | ` learning` | ` learning` | 25 | 25 | 0.8514 |
| 9 | ` century` | ` century` | 20 | 20 | 0.8471 |
| 10 | `.` | `.` | 21 | 21 | 0.8277 |
| 11 | ` avoid` | ` avoid` | 59 | 59 | 0.8125 |
| 12 | ` scaled` | ` scaled` | 28 | 28 | 0.8065 |
| 13 | ` were` | ` techniques` | 27 | 26 | 0.8021 |
| 14 | ` plans` | ` plans` | 53 | 53 | 0.7963 |
| 15 | `human` | `human` | 8 | 8 | 0.7866 |
| 16 | ` default` | ` default` | 39 | 39 | 0.7704 |
| 17 | ` machine` | ` machine` | 24 | 24 | 0.7617 |
| 18 | ` solid` | ` solid` | 52 | 52 | 0.7603 |
| 19 | `.` | `.` | 61 | 61 | 0.7536 |
| 20 | ` or` | ` deceptive` | 45 | 44 | 0.7239 |
| 21 | ` machine` | ` machine` | 9 | 9 | 0.7003 |
| 22 | ` intelligence` | ` intelligence` | 10 | 10 | 0.6972 |
| 23 | ` current` | ` current` | 23 | 23 | 0.6804 |
| 24 | ` significantly` | ` significantly` | 6 | 6 | 0.6803 |
| 25 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.6732 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` to` | ` super` | 58 | 7 | 0.0000 |
| 2 | ` to` | ` that` | 58 | 3 | 0.0000 |
| 3 | `.` | ` significantly` | 61 | 6 | 0.0000 |
| 4 | `.` | ` intelligence` | 61 | 10 | 0.0000 |
| 5 | ` to` | ` that` | 30 | 3 | 0.0000 |
| 6 | ` to` | ` super` | 30 | 7 | 0.0000 |
| 7 | `.` | ` think` | 61 | 2 | 0.0000 |
| 8 | `.` | ` century` | 61 | 20 | 0.0000 |
| 9 | ` to` | ` intelligence` | 58 | 10 | 0.0000 |
| 10 | ` up` | `,` | 29 | 5 | 0.0000 |
| 11 | ` to` | `,` | 58 | 5 | 0.0000 |
| 12 | ` up` | ` If` | 29 | 22 | 0.0000 |
| 13 | ` to` | ` is` | 58 | 11 | 0.0000 |
| 14 | ` up` | ` that` | 29 | 3 | 0.0000 |
| 15 | `.` | ` super` | 61 | 7 | 0.0000 |
| 16 | ` for` | ` super` | 56 | 7 | 0.0000 |
| 17 | ` to` | `human` | 58 | 8 | 0.0000 |
| 18 | ` up` | `human` | 29 | 8 | 0.0000 |
| 19 | ` to` | `,` | 30 | 5 | 0.0000 |
| 20 | ` plans` | ` significantly` | 53 | 6 | 0.0000 |
| 21 | `.` | `human` | 61 | 8 | 0.0000 |
| 22 | `ulative` | `,` | 47 | 5 | 0.0000 |
| 23 | ` to` | ` significantly` | 58 | 6 | 0.0000 |
| 24 | ` manip` | ` that` | 46 | 3 | 0.0000 |
| 25 | ` solid` | `,` | 52 | 5 | 0.0000 |

In [31]:
pct = get_attention_pattern(cache, layer=1, head=0)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L1H0 — {pct:.2f}% ({level})\n\nEnd of text token, itself, token \",\""))

---
## L1H0 — 24.31% (partial)

End of text token, itself, token ","

In [32]:
show_head_pattern(str_tokens, cache, layer=1, head=0)

In [33]:
attention = get_attention_pattern(cache, layer=1, head=0)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.8448 |
| 3 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.7984 |
| 4 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.7369 |
| 5 | `.` | `<\|endoftext\|>` | 21 | 0 | 0.7317 |
| 6 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.7272 |
| 7 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.6945 |
| 8 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.6278 |
| 9 | ` learning` | `<\|endoftext\|>` | 25 | 0 | 0.6088 |
| 10 | ` and` | ` and` | 49 | 49 | 0.4983 |
| 11 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.4949 |
| 12 | ` to` | ` to` | 16 | 16 | 0.4747 |
| 13 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.4548 |
| 14 | `.` | `<\|endoftext\|>` | 61 | 0 | 0.4377 |
| 15 | `human` | `,` | 8 | 5 | 0.4304 |
| 16 | ` techniques` | `<\|endoftext\|>` | 26 | 0 | 0.4184 |
| 17 | ` this` | `<\|endoftext\|>` | 19 | 0 | 0.3891 |
| 18 | ` super` | ` significantly` | 7 | 6 | 0.3312 |
| 19 | `,` | `<\|endoftext\|>` | 33 | 0 | 0.3196 |
| 20 | ` current` | `<\|endoftext\|>` | 23 | 0 | 0.3155 |
| 21 | ` century` | `<\|endoftext\|>` | 20 | 0 | 0.3123 |
| 22 | ` to` | ` to` | 58 | 58 | 0.2926 |
| 23 | ` default` | `<\|endoftext\|>` | 39 | 0 | 0.2897 |
| 24 | ` created` | `<\|endoftext\|>` | 18 | 0 | 0.2895 |
| 25 | `,` | `<\|endoftext\|>` | 48 | 0 | 0.2891 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` to` | ` up` | 58 | 29 | 0.0000 |
| 2 | ` and` | ` up` | 49 | 29 | 0.0000 |
| 3 | ` to` | ` think` | 58 | 35 | 0.0000 |
| 4 | `.` | ` up` | 61 | 29 | 0.0000 |
| 5 | `,` | ` up` | 33 | 29 | 0.0000 |
| 6 | `,` | ` up` | 48 | 29 | 0.0000 |
| 7 | ` to` | ` likely` | 58 | 13 | 0.0001 |
| 8 | ` how` | ` up` | 57 | 29 | 0.0001 |
| 9 | `ulative` | ` think` | 47 | 35 | 0.0001 |
| 10 | `ulative` | ` up` | 47 | 29 | 0.0001 |
| 11 | ` that` | ` up` | 50 | 29 | 0.0001 |
| 12 | ` to` | ` significantly` | 58 | 6 | 0.0001 |
| 13 | ` to` | ` scaled` | 58 | 28 | 0.0001 |
| 14 | ` avoid` | ` up` | 59 | 29 | 0.0001 |
| 15 | ` and` | ` were` | 49 | 27 | 0.0001 |
| 16 | ` to` | ` known` | 58 | 55 | 0.0001 |
| 17 | ` and` | ` is` | 49 | 11 | 0.0001 |
| 18 | `,` | ` likely` | 33 | 13 | 0.0001 |
| 19 | ` current` | ` created` | 23 | 18 | 0.0001 |
| 20 | ` to` | ` created` | 58 | 18 | 0.0001 |
| 21 | ` how` | ` think` | 57 | 35 | 0.0001 |
| 22 | ` how` | ` created` | 57 | 18 | 0.0001 |
| 23 | ` how` | ` scaled` | 57 | 28 | 0.0001 |
| 24 | ` and` | ` be` | 49 | 17 | 0.0001 |
| 25 | ` learning` | ` created` | 25 | 18 | 0.0001 |

In [34]:
pct = get_attention_pattern(cache, layer=1, head=9)[:, 0].mean().item() * 100
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L1H9 — {pct:.2f}% ({level})\n\nDiffuse attention across context, some end of text, broad aggregation"))

---
## L1H9 — 16.82% (partial)

Diffuse attention across context, some end of text, broad aggregation

In [35]:
show_head_pattern(str_tokens, cache, layer=1, head=9)

In [36]:
attention = get_attention_pattern(cache, layer=1, head=9)
show_attention_tables(str_tokens, attention, top_k=25)

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` manip` | `<\|endoftext\|>` | 46 | 0 | 0.6480 |
| 3 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.6478 |
| 4 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.6242 |
| 5 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.6070 |
| 6 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.5625 |
| 7 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.5567 |
| 8 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.5019 |
| 9 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.4962 |
| 10 | ` to` | `<\|endoftext\|>` | 30 | 0 | 0.3982 |
| 11 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.3960 |
| 12 | `We` | `We` | 1 | 1 | 0.3930 |
| 13 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.3223 |
| 14 | `,` | ` think` | 5 | 2 | 0.2979 |
| 15 | ` solid` | `<\|endoftext\|>` | 52 | 0 | 0.2603 |
| 16 | `,` | ` powerful` | 5 | 4 | 0.2375 |
| 17 | ` think` | `We` | 2 | 1 | 0.2300 |
| 18 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.2176 |
| 19 | ` manip` | ` or` | 46 | 45 | 0.2148 |
| 20 | ` powerful` | ` think` | 4 | 2 | 0.2122 |
| 21 | ` significantly` | ` powerful` | 6 | 4 | 0.2031 |
| 22 | ` powerful` | ` powerful` | 4 | 4 | 0.1956 |
| 23 | ` more` | ` powerful` | 12 | 4 | 0.1954 |
| 24 | ` this` | ` machine` | 19 | 9 | 0.1890 |
| 25 | ` current` | ` intelligence` | 23 | 10 | 0.1852 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` manip` | ` would` | 46 | 37 | 0.0001 |
| 2 | ` manip` | ` think` | 46 | 35 | 0.0001 |
| 3 | ` produce` | ` think` | 40 | 35 | 0.0002 |
| 4 | ` plans` | ` think` | 53 | 35 | 0.0002 |
| 5 | ` manip` | ` likely` | 46 | 13 | 0.0002 |
| 6 | ` manip` | ` up` | 46 | 29 | 0.0002 |
| 7 | ` up` | ` up` | 29 | 29 | 0.0002 |
| 8 | ` plans` | ` up` | 53 | 29 | 0.0002 |
| 9 | ` to` | ` think` | 58 | 35 | 0.0002 |
| 10 | ` produce` | ` up` | 40 | 29 | 0.0002 |
| 11 | ` plans` | ` likely` | 53 | 13 | 0.0003 |
| 12 | ` to` | ` would` | 58 | 37 | 0.0003 |
| 13 | ` avoid` | ` would` | 59 | 37 | 0.0003 |
| 14 | ` manip` | ` significantly` | 46 | 6 | 0.0004 |
| 15 | ` produce` | ` likely` | 40 | 13 | 0.0004 |
| 16 | ` avoid` | ` think` | 59 | 35 | 0.0004 |
| 17 | ` up` | ` likely` | 29 | 13 | 0.0004 |
| 18 | ` for` | ` would` | 56 | 37 | 0.0004 |
| 19 | ` for` | ` think` | 56 | 35 | 0.0004 |
| 20 | ` to` | ` likely` | 58 | 13 | 0.0004 |
| 21 | ` to` | ` up` | 30 | 29 | 0.0005 |
| 22 | ` manip` | ` not` | 46 | 15 | 0.0005 |
| 23 | ` this` | ` we` | 60 | 34 | 0.0006 |
| 24 | ` manip` | ` think` | 46 | 2 | 0.0006 |
| 25 | ` known` | ` we` | 55 | 34 | 0.0006 |